In [2]:
"""
Mini-Project - Code 1: F1 Corpus Creation via Web Scraping

Purpose:
This script creates a text corpus by scraping Formula 1 season summaries
from Wikipedia (1950-present). It filters sentences for keywords related
to notable events/accidents and saves the result to a text file.
This file ('f1_notable_sentences_corpus.txt') will be used by the
companion analysis script (Part 2).

Methodology:
- Uses 'requests' and 'BeautifulSoup4' for web scraping Wikipedia.
- Identifies the target season list table using HTML header analysis.
- Extracts URLs for individual F1 season pages.
- Scrapes the lead section of each season page.
- Cleans text (normalizes whitespace).
- Filters sentences using NLTK based on keywords.
- Includes delays between requests.
- Saves the filtered sentences, categorized by season, to the output file.
"""

# --- Import Libraries ---
import requests #  To make HTTP requests for web pages
from bs4 import BeautifulSoup # To parse HTML content
import pandas as pd 
import io # Used to handle text stream for pandas read_html
import time #  To add delays between requests
import re # For regular expressions 
import datetime # To determine the year
import nltk #Natural Language Toolkit for NLP tasks (sent_tokenize)
from urllib.parse import urljoin # To construct full URLs


def get_f1_season_urls(list_url="https://en.wikipedia.org/wiki/List_of_Formula_One_seasons", start_year=1950):
    """
    Finds and returns a dictionary of F1 season URLs ({year: url}) from the main list page on Wikipedia.
    Uses BeautifulSoup to identify the correct table by checking specific required column headers
    ('Season', 'Drivers' Champion') for robustness.

    Features Highlighted:
    - Libraries: 'requests', 'BeautifulSoup', 'pandas', 'urljoin', 're'.
    - HTML Parsing: Identifies table via headers ('<th>').
    - Data Structures: Returns a dictionary mapping year (int) to URL (str).
    - Error Handling: Basic 'try...except'.
    """
    urls = {}
    current_year = datetime.datetime.now().year
    try:
        print(f"Fetching season list from {list_url}")
        # Using a descriptive User-Agent for the request
        headers = {'User-Agent': 'Mozilla/5.0 (Student Project Scraper; Contact: hmm24zfp@bangor.ac.uk)'}
        response = requests.get(list_url, headers=headers, timeout=15)
        response.raise_for_status() # Ensure the page was successfully fetched

        # Parse the full HTML with BeautifulSoup to find the table and extract links
        soup = BeautifulSoup(response.content, 'lxml')
        target_table = None
        all_tables = soup.find_all('table', class_='wikitable') # Look for standard Wikipedia tables
        # Define the header text to ensure we have the right table
        expected_headers_check = {'season', "drivers' champion"}

        print(f"Found {len(all_tables)} wikitables.")
        # Iterate through found wikitables and check their HTML headers
        for table in all_tables:
            header_row = table.find('tr') #got this from inspecting the wiki webpage for rows
            if not header_row: continue
            header_cells = header_row.find_all('th') #got this from inspecting the wiki webpage for cells
            if not header_cells: continue
            # Extract and normalize header text for comparison
            header_texts = {th.get_text(strip=True).lower() for th in header_cells}
            # Check if this table contains the essential headers
            is_target = all(any(expected in h for h in header_texts) for expected in expected_headers_check) # checking if all conatin the expected headers if yes they are, then count+1
            if is_target:
                 target_table = table
                 print("Identified target HTML table based on headers.")
                 break # Use the first table that matches

        if not target_table:
            # Raise error if the specific table wasn't found after checking headers
            raise ValueError("Could not find the specific HTML table with required headers.")

        # Extract year and URL from the rows of the confirmed table
        rows = target_table.find_all('tr')
        for row in rows[1:]: # Skip the header row
            cells = row.find_all(['td', 'th'], limit=1) # Only need the first cell
            if not cells: continue
            year_text = cells[0].get_text(strip=True)
            year_match = re.match(r'^(\d{4})', year_text) # Extract year with Regular Functions
            if year_match:
                year = int(year_match.group(1))
                # Collect URL if it's within the specified year range
                if start_year <= year <= current_year:
                    link = cells[0].find('a', href=re.compile(r'^/wiki/')) #For wiki links
                    if link:
                        full_url = urljoin(list_url, link['href']) # Construct full URL
                        # Store the first valid URL found for the year, avoid self-links
                        if year not in urls and list_url not in full_url: urls[year] = full_url

        print(f"Found {len(urls)} season URLs.")
        return urls
    except Exception :
        # Report errors encountered during URL fetching
        print(f"ERROR getting season URLs: {Exception}")
        return {}

def scrape_and_filter_lead(url, keywords_lower):
    """
    Scrapes the lead section text (first few paragraphs) from a given Wikipedia URL,
    cleans it by removing citations and normalizing whitespace, tokenizes it into sentences (NLTK),
    and returns a list of sentences containing any of the specified keywords.

    Features Highlighted:
    - Text Extraction: Uses 'BeautifulSoup.stripped_strings' generator.
    - Text Cleaning: Uses 're.sub' with regular expressions.
    - NLP (NLTK): Uses 'nltk.sent_tokenize'. 'Punkt' toolkit required and used
    - List Manipulation: Appends results to list.
    - String Comparison: Case-insensitive keyword check ('in').
    """
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Student Project Scraper; Contact: hmm24zfp@bangor.ac.uk )'}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status() # Check page fetch status
        soup = BeautifulSoup(response.content, 'lxml')
        # Locate the main content area
        content_div = soup.find('div', {'class': 'mw-parser-output'})
        if not content_div: return []

        # Extract text from first 10 paragraphs 
        paragraphs_text = []
        limit = 10
        # Feature: Iteration with limit
        for element in content_div.find_all('p', recursive=False):
             if len(paragraphs_text) >= limit: break
             # Feature: Joining strings from generator with space separator
             para_text = " ".join(element.stripped_strings)
             if para_text: paragraphs_text.append(para_text)

        lead_text = " ".join(paragraphs_text)
        lead_text = re.sub(r'\[.*?\]', '', lead_text) #  Remove citations and unnecessary punctuations
        lead_text = re.sub(r'\s+', ' ', lead_text).strip() # Normalize whitespace

        if not lead_text: 
            return []


        # Tokenize into sentences and filter based on keywords
        sentences = nltk.sent_tokenize(lead_text)
        relevant_sentences = []
        for sentence in sentences:
            cleaned = sentence.strip()
            if len(cleaned) < 15: continue # Ignore very short sentences
            sentence_lower = cleaned.lower()
            # check if any keyword exists in the sentence
            # check using 'any' and generator expression
            if any(keyword in sentence_lower for keyword in keywords_lower):
                relevant_sentences.append(cleaned)

        return relevant_sentences

    except Exception:
        # Report errors during scraping/filtering of a single page
        # Allows the main loop to continue with other pages
        print(f"  Skipping {url.split('/')[-1]} due to error: {Exception}")
        return []


""" MAIN EXECUTION """


if __name__ == "__main__": #code runs only when script is executed directly
    print("Starting F1 Corpus Creation Script")

    # Keywords used to filter sentences
    KEYWORDS = [
        "accident", "crash", "collision", "incident", "controversy", "protest",
        "penalty", "penalised", "penalized", "disqualified", "fatal", "injured",
        "injury", "record", "maiden", "first win", "debut", "retired", "champion",
        "clinch", "decider", "controversial", "unsafe", "failure", "tyre", "tire",
        "engine", "suspension", "gearbox", "safety car", "red flag", "investigation",
        "wet race", "strategy", "overtake", "dominant", "comeback", "historic", "scandal"
    ]
    OUTPUT_FILENAME = "f1_notable_sentences_corpus.txt" # Corpus file
    DELAY = 1.5 # Delay in seconds between requests
    START_YEAR = 1950

    # Get Season URLs 
    # Calls the function defined above to get URLs from 1950 to present
    season_urls = get_f1_season_urls(start_year=START_YEAR)

    #Scrape and Collect Sentences 
    all_output_lines = [] # List to store collected headers and sentences
    processed_ok_count = 0 # Counter for seasons for relevant sentences
    error_skip_count = 0 # Counter for seasons not included due to errors or no sentences

    # Proceed only if URLs were actually found
    if not season_urls:
        print("No Season URLs were found. Cannot create corpus.")
    else:
        # using dictionary for sorting
        years_to_process = sorted(season_urls.keys())
        total_seasons = len(years_to_process)
        print(f"Starting to scrape for {total_seasons} seasoms")

        # Loop through each year and its corresponding URL
        # Use of enumerate to get index and value
        for i, year in enumerate(years_to_process):
            url = season_urls[year]
            # Display progress periodically using increments of 20
            if (i + 1) % 20 == 0 or i == 0 : 
                print(f"  Processing {year} ({i+1}/{total_seasons})")

            # Scrape the lead section and filter sentences for keywords
            filtered_sentences = scrape_and_filter_lead(url, KEYWORDS)

            # If relevant sentences were found, add them to the collection
            if filtered_sentences:
                processed_ok_count += 1
                all_output_lines.append(f"\n{year} Season\n") # Add a header for the year
                all_output_lines.extend(filtered_sentences) # Add the list of sentences
            else:
                # Increment skip count if scraping failed or no relevant sentences found
                error_skip_count += 1

            # Pause between requests to be respectful to the server
            # Feature: Use of time.sleep() for rate limiting
            time.sleep(DELAY)

        # Write Corpus File
        print("\nWriting collected data to corpus file")
        if all_output_lines:
            try:
                # Prepare informational header for the output file
                file_header = "# F1 Season Notable Events Corpus\n"
                file_header += "# Generated by scraping Wikipedia lead sections.\n"
                file_header += f"# Keywords Used for Filtering: {', '.join(KEYWORDS)}\n\n"

                # Combine header and all collected lines
                corpus_content = file_header + "\n".join(all_output_lines)

                # Write to file using a context manager for safety
                # Feature Highlight: Context Manager 'with open' ensures file closure.
                with open(OUTPUT_FILENAME, 'w', encoding='utf-8') as f:
                    f.write(corpus_content)

                # Calculate number of actual sentences written
                sentence_count = sum(1 for line in all_output_lines if not line.startswith('---') and line.strip())
                print(f"Successfully saved {sentence_count} sentences to {OUTPUT_FILENAME}.")

            except Exception as e:
                print(f"ERROR writing corpus file: {e}")
        else:
            # Message if no data was collected from any season
            print("No relevant sentences were collected, file not written.")

    print("\nCorpus Creation Finished")
    print(f"Seasons Attempted: {len(season_urls) if season_urls else 0}")
    print(f"Seasons Yielding Sentences: {processed_ok_count}")
    print(f"Seasons Skipped or No Relevant Sentences: {error_skip_count}")

Starting F1 Corpus Creation Script
Fetching season list from https://en.wikipedia.org/wiki/List_of_Formula_One_seasons
Found 2 wikitables.
Identified target HTML table based on headers.
Found 76 season URLs.
Starting to scrape for 76 seasoms
  Processing 1950 (1/76)
  Processing 1969 (20/76)
  Processing 1989 (40/76)
  Processing 2009 (60/76)

Writing collected data to corpus file
Successfully saved 1235 sentences to f1_notable_sentences_corpus.txt.

Corpus Creation Finished
Seasons Attempted: 76
Seasons Yielding Sentences: 70
Seasons Skipped or No Relevant Sentences: 6


In [3]:
"""
CODE 2 : PERFORMING SPACY ANALYSIS ON SCRAPED TEXT/CORPUS

This script analyzes the text corpus created by the companion scraping
script ('f1_notable_sentences_corpus.txt'). It performs Natural Language
Processing tasks using NLTK (for Part-of-Speech tagging) and spaCy
(for Named Entity Recognition) as required by the mini-project brief.
It also calculates basic corpus statistics (sentences, words, characters).

Methodology:
- Loads the corpus text from the specified file.
- Calculates basic statistics (sentences, words, characters).
- Uses NLTK for word tokenization and POS tagging, then analyzes tag frequencies.
- Uses Spacy to load a pre-trained model ('en_core_web_sm'), perform NER,
  and analyze entity frequencies (PERSON, ORG, GPE).
- Prints summary results of the analyses and statistics.
"""


import nltk 
import spacy 
from collections import Counter 
import re
import os


def load_corpus_text(filename):
    """
    Features Highlighted:
    - Uses 'with open', specifies 'utf-8' encoding. Alos uses os library to confirm if file exists or handles if not.
    - Using 'try and except FileNotFoundError' error handling for efficiency.
    - Filters lines starting with '#'.
    - Uses 're.sub' to remove the season marker separators.
    - '.join()' combines lines into single text block.
    """
    # Check if the corpus exists before trying to open it
    if not os.path.exists(filename):
         print(f"ERROR: Corpus file '{filename}' not found.")
         return None
    try:
        print(f"Loading corpus text from '{filename}'")
        with open(filename, 'r', encoding='utf-8') as f:
            # Read lines and filter out comment/header lines starting with '#'
            # List comprehension for filtering
            lines = [line for line in f if not line.strip().startswith('#')]
        # Join lines, then remove the 'YEAR Season' separators for cleaner text analysis
        full_text = "".join(lines)
        # Regular expression substitution
        full_text = re.sub(r"\n---\s*\d{4}\s*Season\s*---\n", "\n", full_text) # Replace headers with newline
        return full_text.strip() # Return final cleaned text
    except Exception:
        print(f"ERROR reading corpus file '{filename}': {Exception}")
        return None

def analyze_corpus_pos(corpus_text, top_n=15): # predefining top_n to a value to take that many values i.e., top 15
    """
    Performs Part-of-Speech (POS) tagging and frequency analysis using NLTK.

    Features Highlighted:
    - Uses 'nltk.word_tokenize', 'nltk.pos_tag' for tokenising and tagging.
    - Uses 'collections.Counter' for efficient frequency counting.
    - Uses set values (`tag in noun_tags`) and conditionals (`len(word) > 2`).
    - Prints results using f-strings.
    """
    if not corpus_text: 
        print(" No corpus text provided."); return

    
    try:
        words = nltk.word_tokenize(corpus_text)
        if not words: print("No words found after tokenization."); return
        tagged_words = nltk.pos_tag(words)

    
        # Overall POS Tag Frequency
        # Use of collections.Counter with generator expression
        pos_counts = Counter(tag for word, tag in tagged_words)
        print(f"\nTop {top_n} most common Part-of-Speech tags:\n")
        for tag, count in pos_counts.most_common(top_n): print(f"{tag:<5}: {count}")

        # Frequency of specific word types (Nouns, Verbs)
        noun_tags = {'NN', 'NNS', 'NNP', 'NNPS'}
        noun_counts = Counter(word.lower() for word, tag in tagged_words if tag in noun_tags and len(word) > 2)
        print(f"\nTop {top_n} most common Nouns:\n")
        for noun, count in noun_counts.most_common(top_n): print(f"{noun}: {count}")

        verb_tags = {'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ'}
        verb_counts = Counter(word.lower() for word, tag in tagged_words if tag in verb_tags and len(word) > 2)
        print(f"\nTop {top_n} most common Verbs:\n")
        for verb, count in verb_counts.most_common(top_n): print(f"{verb}: {count}")

    except Exception:
        print(f"ERROR during POS analysis: {Exception}")

def analyze_corpus_ner(corpus_text, top_n=15, model="en_core_web_sm"):
    """
    Performs Named Entity Recognition (NER) using Spacy to identify and count entities.

    Features Highlighted:
    - Uses 'spacy.load()', 'nlp()', 'doc.ents', '.text', '.label_'.
    - Example filters by entity label (e.g., `ent.label_ == 'PERSON'`).
    - Includes check for text length relative to Spacy model limits.
    """
    if not corpus_text: print(" No corpus text provided "); return

    print("\nStarting Named Entity Recognition (NER) Analysis using Spacy")
    try:
        nlp = spacy.load(model)
        max_spacy_len = nlp.max_length # Get model's limit
        doc = None # Initialize doc
        if len(corpus_text) > max_spacy_len:
            print(f"Warning: Corpus > {max_spacy_len:,} chars (spaCy limit). Analyzing first {max_spacy_len} chars.")
            corpus_text_subset = corpus_text[:max_spacy_len]
            doc = nlp(corpus_text_subset)
        else:
            doc = nlp(corpus_text)

        if not doc.ents:
            print("No named entities found by Spacy."); return
            
        print(f"\nFound {len(doc.ents)} total named entities.")

        # Frequency of Entity Labels (Types)
        label_counts = Counter(ent.label_ for ent in doc.ents)
        print(f"\nTop {top_n} most common Entity Types:\n")
        for label, count in label_counts.most_common(top_n): print(f"{label}: {count}")

        #Frequency of specific entity texts (Persons, Organizations, Locations)
        person_counts = Counter([ent.text.strip() for ent in doc.ents if ent.label_ == 'PERSON'])
        print(f"\nTop {top_n} most common PERSON entities:\n")
        for entity, count in person_counts.most_common(top_n): print(f"{entity}: {count}")

        org_counts = Counter([ent.text.strip() for ent in doc.ents if ent.label_ == 'ORG'])
        print(f"\nTop {top_n} most common ORG entities:\n")
        for entity, count in org_counts.most_common(top_n): print(f"{entity}: {count}")

        gpe_counts = Counter([ent.text.strip() for ent in doc.ents if ent.label_ == 'GPE'])
        print(f"\nTop {top_n} most common GPE entities:\n")
        for entity, count in gpe_counts.most_common(top_n): print(f"{entity}: {count}")

    except Exception:
        print(f"ERROR during NER analysis: {Exception}")

""" MAIN SCRIPT EXECUTION """

if __name__ == "__main__":
    print("// Starting F1 Corpus Analysis Script //")

    # Define the corpus file created by the scraping script
    CORPUS_FILENAME = "f1_notable_sentences_corpus.txt" 

   
    corpus_full_text = load_corpus_text(CORPUS_FILENAME)

    
    if corpus_full_text:
        print(f"Corpus loaded successfully ({len(corpus_full_text)} characters).") # checks if corpus is loaded successfully

        
        
        sentence_count, word_count, char_count = 0, 0, 0 # Initialize
        try:
            print("\n// Calculating Corpus Statistics //")

            corpus_sentences = nltk.sent_tokenize(corpus_full_text)
            corpus_words = nltk.word_tokenize(corpus_full_text)
            sentence_count = len(corpus_sentences)
            word_count = len(corpus_words)
            char_count = len(corpus_full_text) # Includes spaces

            print(f"Sentences (NLTK tokenized): {sentence_count}")
            print(f"Words (NLTK tokenized): {word_count}")
            print(f"Characters (including spaces): {char_count}")

        except Exception:
            print(f"ERROR calculating corpus stats: {Exception}")


        
        print("\n// NLP Analysis //")

        # Perform POS Tag Analysis using NLTK
        analyze_corpus_pos(corpus_full_text)

        # Perform Named Entity Recognition using Spacy
        analyze_corpus_ner(corpus_full_text)

    else:
        # Message if corpus file wasn't found or couldn't be read
        print("Corpus analysis could not proceed.")

// Starting F1 Corpus Analysis Script //
Loading corpus text from 'f1_notable_sentences_corpus.txt'
Corpus loaded successfully (167218 characters).

// Calculating Corpus Statistics //
Sentences (NLTK tokenized): 1165
Words (NLTK tokenized): 30480
Characters (including spaces): 167218

// NLP Analysis //

Top 15 most common Part-of-Speech tags:

NNP  : 5447
IN   : 3525
NN   : 3415
DT   : 2920
VBD  : 1973
JJ   : 1880
CD   : 1423
,    : 1337
.    : 1165
NNS  : 1164
CC   : 973
RB   : 805
VBN  : 692
VBG  : 597
TO   : 493

Top 15 most common Nouns:

championship: 566
world: 365
season: 307
drivers: 306
formula: 299
one: 206
prix: 150
race: 149
fia: 147
champion: 143
grand: 133
ferrari: 122
teams: 121
engine: 118
constructors: 116

Top 15 most common Verbs:

was: 427
won: 145
had: 139
were: 134
competed: 84
retired: 79
contested: 70
featured: 61
took: 57
started: 40
crashed: 37
ended: 33
reigning: 32
been: 32
win: 31

Starting Named Entity Recognition (NER) Analysis using Spacy

Found 4810 t